# Day 9 v2 — Silver: Invoice Metadata (For Loop Pattern)

**Same logic as v1 but wrapped in an entity config list.**

**What changed from v1:**
- Entity config list with natural_key
- Wrapped in a for loop
- Full overwrite (incremental MERGE is v3)

**Bronze source:** `bronze-volume/invoices/YYYY/MM/DD/*.pdf` (raw PDFs — scanned via `dbutils.fs`)
**Silver sink:** `silver-volume/invoices/` (Delta)

Note: Bronze stores only raw PDFs. No `_metadata` Delta table exists.
Metadata (invoice_id, year, month, day, file_size_kb) is extracted from file paths.

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_BATCH_BASE = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
SILVER_BATCH_BASE = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze base : {BRONZE_BATCH_BASE}")
print(f"Silver base : {SILVER_BATCH_BASE}")
print(f"Run time    : {RUN_TS}")

In [ ]:
# Cell 3 — Entity config for blob batch entities
# Bronze format: raw PDFs under invoices/YYYY/MM/DD/ — no Delta metadata table.
# Silver derives metadata from file path + size.

ENTITIES = [
    {
        "entity_name"  : "invoices",
        "natural_key"  : "invoice_id",
        "bronze_path"  : f"{BRONZE_BATCH_BASE}/invoices",   # root PDF folder
        "silver_path"  : f"{SILVER_BATCH_BASE}/invoices",
    },
]

print(f"Batch entities configured: {len(ENTITIES)}")
for e in ENTITIES:
    print(f"  {e['entity_name']:<20} key={e['natural_key']}  bronze={e['bronze_path']}")

In [ ]:
# Cell 4 — For loop: transform all batch entities

import re
from pyspark.sql import Row

def list_pdfs(path):
    results = []
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return results
    for item in items:
        if item.isDir():
            results.extend(list_pdfs(item.path))
        elif item.name.endswith(".pdf"):
            results.append(item)
    return results

results = []

for entity in ENTITIES:
    name        = entity["entity_name"]
    natural_key = entity["natural_key"]
    bronze_path = entity["bronze_path"]
    silver_path = entity["silver_path"]

    print(f"Processing: {name} ...", end=" ")

    try:
        # Step 1: Scan Bronze PDF files and extract metadata from file paths
        pdf_files = list_pdfs(bronze_path)
        rows = []
        for f in pdf_files:
            m = re.search(r"invoices/(\d{4})/(\d{2})/(\d{2})/(.+\.pdf)$", f.path)
            if m:
                rows.append(Row(
                    invoice_id   = m.group(4).replace(".pdf", ""),
                    year         = m.group(1),
                    month        = m.group(2),
                    day          = m.group(3),
                    file_size_kb = round(f.size / 1024, 1),
                    bronze_path  = f.path,
                ))

        bronze_rows = len(rows)
        if bronze_rows == 0:
            raise Exception(f"No PDF files matched expected path pattern under {bronze_path}.")

        raw_df = spark.createDataFrame(rows)

        # Step 2: Cast and enrich
        typed_df = (
            raw_df
            .withColumn("invoice_id",    F.trim(F.col("invoice_id")))
            .withColumn("invoice_year",  F.col("year").cast("integer"))
            .withColumn("invoice_month", F.col("month").cast("integer"))
            .withColumn("invoice_day",   F.col("day").cast("integer"))
            .withColumn("file_size_kb",  F.col("file_size_kb").cast("decimal(10,2)"))
            .withColumn("bronze_path",   F.trim(F.col("bronze_path")))
            .drop("year", "month", "day")
            .withColumn(
                "invoice_date",
                F.to_date(
                    F.concat_ws("-",
                        F.col("invoice_year").cast("string"),
                        F.lpad(F.col("invoice_month").cast("string"), 2, "0"),
                        F.lpad(F.col("invoice_day").cast("string"),   2, "0"),
                    ), "yyyy-MM-dd"
                )
            )
            .withColumn(
                "invoice_number",
                F.regexp_extract(F.col("invoice_id"), r"(\d+)$", 1).cast("long")
            )
        )

        # Step 3: Add audit columns
        audited_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit("full"))
            .withColumn("silver_pipeline",    F.lit("pl_silver_blob_invoices_v2"))
        )

        # Step 4: Deduplicate on natural_key
        window = Window.partitionBy(natural_key).orderBy(F.col("silver_ingested_at").desc())
        deduped_df = (
            audited_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )
        silver_rows = deduped_df.count()

        # Step 5: Write to Silver Delta (full overwrite)
        (
            deduped_df.write.format("delta")
            .mode("overwrite").option("overwriteSchema", "true")
            .save(silver_path)
        )

        print(f"OK  (bronze={bronze_rows} -> silver={silver_rows})")
        results.append({"entity": name, "status": "succeeded",
                        "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as ex:
        print(f"FAILED")
        results.append({"entity": name, "status": "failed",
                        "bronze_rows": 0, "silver_rows": 0, "error": str(ex)})

print("\nAll batch entities done.")

In [ ]:
# Cell 5 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 60)
print("SILVER BLOB INVOICES v2 — RUN SUMMARY")
print("=" * 60)
print(f"  run_timestamp : {RUN_TS}")
print(f"  succeeded     : {len(succeeded)}")
print(f"  failed        : {len(failed)}")
print()

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['entity']:<20} bronze={r['bronze_rows']:>8}  silver={r['silver_rows']:>8}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"\nAll {len(succeeded)} entities written to Silver successfully.")